<a href="https://colab.research.google.com/github/junseo25/ds2002-fa26/blob/main/notebooks/04-json-apis%20/2026_10_09_Jun_Seo_Lee_%E2%80%94_Public_API_to_Clean_Table_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [29]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }

    # TODO: loop up to `tries` times
    last_error = None

    for attempt in range(tries):

        # TODO: request with a timeout; return a DataFrame on 200
        try:
            r = requests.get(BASE, params=params, timeout=15)

        except requests.RequestException as err:
            last_error = repr(err)

        else:
            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])

            # TODO: raise on a status not in RETRY_STATUSES
            if r.status_code not in RETRY_STATUSES:
                raise RuntimeError(
                    f"Not retryable: {r.status_code}: {r.text[:200]}"
                )

            last_error = f"{r.status_code}: {r.text[:200]}"

        # TODO: sleep (increasing each time) and try again
        if attempt < tries - 1:
            delay = 0.5 * (2 ** attempt)
            print(
                f"Attempt {attempt + 1} failed ({last_error}); "
                f"retrying in {delay}s"
            )
            time.sleep(delay)

    # TODO: raise if every attempt failed
    raise RuntimeError(
        f"Gave up after {tries} attempts: {last_error}"
    )

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [30]:
# TODO: call get_weather once for a range that spans both dates,
#       then keep just the two dates you care about
used_fallback = False

wx = get_weather('2026-09-05', '2026-10-03')

wx = wx.loc[
    wx['time'].isin(['2026-09-05', '2026-10-03'])
].copy()

wx

,time,temperature_2m_max,precipitation_sum
0,2026-09-05,29.4,2.5
28,2026-10-03,22.0,14.7


### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [31]:
# TODO: call once, keeping only the two dates
used_fallback = False

try:
    wx = get_weather('2026-09-05', '2026-10-03')
    wx = wx.loc[
        wx['time'].isin(['2026-09-05', '2026-10-03'])
    ].copy()

except (requests.RequestException, RuntimeError) as err:
    # TODO: set used_fallback and build wx from FALLBACK
    print(f"Weather fetch failed; using fallback sample: {err}")
    used_fallback = True
    wx = FALLBACK.copy()

# TODO: add a source column
wx['source'] = 'fallback sample' if used_fallback else 'open-meteo'

wx

,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [32]:
# TODO
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})

# TODO: make date a real datetime column
wx['date'] = pd.to_datetime(wx['date'])

# TODO: keep the required columns in order
wx = wx[['date', 'temp_max', 'precip_mm', 'source']].copy()

wx

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [33]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [34]:
# TODO
wettest = wx.loc[wx['precip_mm'].idxmax()]
driest = wx.loc[wx['precip_mm'].idxmin()]
difference = wettest['precip_mm'] - driest['precip_mm']

source = 'fallback sample data' if used_fallback else 'Open-Meteo data'

if difference == 0:
    print(
        f"Using {source}, both dates had the same rainfall: "
        f"{wettest['precip_mm']:.1f} mm."
    )
else:
    print(
        f"Using {source}, {wettest['date']:%B %d, %Y} was wetter "
        f"than {driest['date']:%B %d, %Y} by {difference:.1f} mm."
    )

Using Open-Meteo data, October 03, 2026 was wetter than September 05, 2026 by 12.2 mm.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [35]:
# TODO
from datetime import datetime
from zoneinfo import ZoneInfo

print('Endpoint:', BASE)
print('Coordinates: latitude=38.03, longitude=-78.51')
print('Requested date range: 2026-09-05 to 2026-10-03')
print('Selected dates: 2026-09-05 and 2026-10-03')

print('Source:', 'fallback sample' if used_fallback else 'open-meteo')

print(
    'Citation recorded:',
    datetime.now(ZoneInfo('America/New_York')).isoformat(timespec='seconds')
)

pulled_at = pd.Timestamp.now(tz='America/New_York')

print('Pulled:', 'No successful retrieval; used fallback'
      if used_fallback else pulled_at.isoformat())

Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: latitude=38.03, longitude=-78.51
Requested date range: 2026-09-05 to 2026-10-03
Selected dates: 2026-09-05 and 2026-10-03
Source: open-meteo
Citation recorded: 2026-10-11T23:59:08-04:00
Pulled: 2026-10-11T23:59:08.645215-04:00


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

A left join would keep the sales days but leave information such as the weather values missing. Filling these values with something like zeros would falsely insinuate information that the day was dry. Meanwhile, leaving these values as missing may cause the plot to omit the observations.

The failure could go unnoticed because returning an empty table would hide the original API error.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [36]:
# stretch